# 04 — Attention with the mask off

In [buka](https://github.com/harrisonstark/buka) a causal mask stops position 0 from reading the future, because the future is the answer being predicted.

Here the message is already written. Position 0 is allowed to look at the end of the line. That single change is why one forward pass can answer a question about the whole state.

`key_mask` is 1 on real tokens and 0 on pad. Pad keys get a large negative score so softmax ignores them.

**You implement:** `bidirectional_attention`. Tensors are `(batch, heads, time, dim)`.



In [ ]:
from __future__ import annotations

import torch
from torch import Tensor


def bidirectional_attention(q: Tensor, k: Tensor, v: Tensor, key_mask: Tensor | None = None) -> Tensor:
    raise NotImplementedError("bidirectional_attention")



<details>
<summary><b>Solution</b> (try yourself first — click to expand)</summary>

```python
import torch
from torch import Tensor

def bidirectional_attention(q: Tensor, k: Tensor, v: Tensor, key_mask: Tensor | None = None) -> Tensor:
    scale = q.size(-1) ** -0.5
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    if key_mask is not None:
        keep = key_mask.to(dtype=scores.dtype)
        scores = scores + (1.0 - keep)[:, None, None, :] * -1.0e9
    weights = torch.softmax(scores, dim=-1)
    return torch.matmul(weights, v)
```

Copy into the stub cell above, then run **Check**.

</details>



## Check



In [ ]:
q = torch.zeros(1, 1, 3, 2)
k = torch.zeros(1, 1, 3, 2)
v = torch.tensor([0.0, 0.0, 1.0, 1.0, 2.0, 2.0]).view(1, 1, 3, 2)
out = bidirectional_attention(q, k, v)
assert abs(float(out[0, 0, 0, 0]) - 1.0) < 1e-4, float(out[0, 0, 0, 0])
masked = bidirectional_attention(q, k, v, key_mask=torch.tensor([[1.0, 1.0, 0.0]]))
assert abs(float(masked[0, 0, 0, 0]) - 0.5) < 1e-4
print("position 0 saw the future:", float(out[0, 0, 0, 0]))

